In [1]:
%%writefile main.py

from __future__ import annotations

import dataclasses
import os
import sys
from dataclasses import dataclass

# Make the sibling ``orbit_lite`` package importable wherever this file runs:
# loaded in place, dropped at a submission-archive root, or exec'd by
# kaggle_environments with no ``__file__`` (fall back to the working dir).
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _HERE = os.getcwd()
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

import torch
from torch import Tensor

from orbit_lite.geometry import fleet_speed
from orbit_lite.intercept_aim import intercept_angle
from orbit_lite.movement import MovementConfig, PlanetMovement
from orbit_lite.movement_step import (
    apply_private_planned_launches,
    concat_launch_entries,
    disambiguate_duplicate_launches,
    ensure_planet_movement,
    infer_planned_launches_from_entries,
)
from orbit_lite.obs import parse_obs
from orbit_lite.distance_cache import build_distance_cache
from orbit_lite.planner_core import (
    _candidate_indices,
    _empty_entries,
    _greedy_select,
    _plan_regroup,
    build_target_shortlist,
    capture_floor,
    empty_action_row,
    entries_to_sparse_payload,
    largest_initial_player_count,
    make_launch_set,
    reachable_mask,
    reinforcement_timing_factor,
    safe_drain,
    score_candidates,
)
from orbit_lite.adapter import single_obs_to_tensor, sparse_action_row_to_moves


@dataclass(frozen=True)
class ProducerLiteConfig:
    """Behaviour knobs.  """

    
    # the projection window, the movement build length, AND the target ETA cap 
    horizon: int = 18
    # --- shortlists ------------------------------------------------------
    max_sources_per_lane: int = 12
    max_offensive_targets: int = 12         # enemy/neutral proximity targets
    max_defensive_targets: int = 4          
    # --- scoring / greedy ------------------------------------------------
    max_waves_per_turn: int = 6
    roi_threshold: float = 1.5              # fire if score > this
    min_ships_to_launch: float = 4.0
    ship_cost_penalty: float = 0.015
    overkill_penalty: float = 0.04
    neutral_overkill_penalty: float = 0.055
    defense_ship_cost_penalty: float = 0.004
    min_capture_margin: float = 1.0
    neutral_rush_bonus: float = 0.22
    neutral_prod_bonus: float = 0.10
    neutral_eta_penalty: float = 0.025
    neutral_early_turn_limit: int = 70
    # --- regroup  ------------------------------
    enable_regroup: bool = True
    max_regroup_time: float = 7.0
    regroup_pressure_delta_min: float = 0.25
    max_regroup_sources_per_lane: int = 6
    max_regroup_targets_per_source: int = 7
    regroup_pressure_norm: str = "none"
    regroup_time_penalty_weight: float = 1e-3


def _movement_config(config: ProducerLiteConfig, *, player_count: int) -> MovementConfig:
    """MovementConfig: fleet tracking on, horizon = config.horizon."""
    return MovementConfig(
        movement_horizon=int(config.horizon),
        drift_epsilon=1e-3,
        track_fleets=True,
        player_count=int(player_count),
        max_tracked_fleets=128,
    )


_DEBUG_FLEET_FIELDS_PRINTED = False


def _debug_print_fleet_fields(obs, obs_tensors: dict) -> None:
    """One-shot Kaggle log probe for fleet target/in-flight schema."""
    global _DEBUG_FLEET_FIELDS_PRINTED
    if _DEBUG_FLEET_FIELDS_PRINTED:
        return
    _DEBUG_FLEET_FIELDS_PRINTED = True
    try:
        print("[orbit_probe] begin fleet schema probe", flush=True)
        obs_keys = sorted(list(vars(obs).keys())) if hasattr(obs, "__dict__") else []
        print(f"[orbit_probe] obs_keys={obs_keys}", flush=True)
        fleet_like = [k for k in obs_keys if k.startswith("f_") or "fleet" in k.lower() or "target" in k.lower()]
        print(f"[orbit_probe] obs_fleet_like_keys={fleet_like}", flush=True)
        for name in fleet_like:
            val = getattr(obs, name)
            shape = tuple(val.shape) if hasattr(val, "shape") else None
            dtype_name = str(val.dtype) if hasattr(val, "dtype") else type(val).__name__
            if hasattr(val, "reshape"):
                sample = val.reshape(-1)[:8].detach().cpu().tolist()
            else:
                sample = val
            print(f"[orbit_probe] obs.{name}: shape={shape} dtype={dtype_name} sample={sample}", flush=True)
        tensor_keys = sorted(list(obs_tensors.keys()))
        print(f"[orbit_probe] obs_tensors_keys={tensor_keys}", flush=True)
        tensor_fleet_like = [k for k in tensor_keys if "fleet" in str(k).lower() or "target" in str(k).lower()]
        print(f"[orbit_probe] tensor_fleet_like_keys={tensor_fleet_like}", flush=True)
        for name in tensor_fleet_like:
            val = obs_tensors[name]
            shape = tuple(val.shape) if hasattr(val, "shape") else None
            dtype_name = str(val.dtype) if hasattr(val, "dtype") else type(val).__name__
            if hasattr(val, "reshape"):
                sample = val.reshape(-1)[:16].detach().cpu().tolist()
            else:
                sample = val
            print(f"[orbit_probe] obs_tensors[{name!r}]: shape={shape} dtype={dtype_name} sample={sample}", flush=True)
        print("[orbit_probe] end fleet schema probe", flush=True)
    except Exception as exc:
        print(f"[orbit_probe] fleet schema probe failed: {type(exc).__name__}: {exc}", flush=True)


def cheap_enemy_pressure(obs, cache, *, horizon: float, player_id: int) -> Tensor:
    """Reachable-enemy-mass proxy per planet — ``[P]``.

    v1-flying-pressure: original baseline + in-flight enemy fleet contribution.
    Pressure on planet ``t`` = sum over enemy sources (planets and fleets) of
    ``ships * decay``, where ``decay = (1 - d / reach_dist).clamp(0, 1)``.

    Source garrison term (unchanged from v1):
      d = cross_dist[0][src, tgt]
      reach_dist = fleet_speed(src.ships) * horizon

    NEW — in-flight enemy fleet term:
      For each alive enemy fleet f (owner != player_id), compute Euclidean
      distance from fleet's current (x,y) to each planet t's current (x,y),
      and apply the same decay structure with the fleet's own speed.
      A fleet that's already 70% of the way to me carries more weight than
      its origin planet, which is the whole point.

    Notes:
      - Approximations still in: ignores target orbital drift over horizon,
        production accrued in flight, per-owner split.
      - Fleet angle is ignored (treated as "could be heading anywhere") —
        upper-bound proxy. Conservative: overstates pressure for fleets
        flying away from t. Acceptable as a regroup gradient (it's a rank).
    """
    P = int(obs.P)
    device = obs.device
    dtype = obs.ships.dtype
    if P == 0:
        return torch.zeros(P, dtype=dtype, device=device)
    pid = int(player_id)
    H = max(float(horizon), 1e-6)

    # ----- planet-source term (unchanged) ---------------------------------
    d0 = cache.cross_dist[0].to(dtype)                                   # [src, tgt]
    ships = obs.ships.to(dtype)
    speeds = fleet_speed(ships.clamp(min=1e-6))                          # [P]
    reach_dist = (speeds.view(P, 1) * H).clamp(min=1e-6)                 # [src, 1]
    enemy = obs.alive & (obs.owner_abs >= 0) & (obs.owner_abs != pid)    # [P]
    eye = torch.eye(P, device=device, dtype=torch.bool)
    valid = enemy.view(P, 1) & obs.alive.view(1, P) & ~eye               # [src, tgt]
    decay = (1.0 - d0 / reach_dist).clamp(min=0.0)
    contrib_planets = torch.where(valid, ships.view(P, 1) * decay, torch.zeros_like(decay))
    pressure = contrib_planets.sum(dim=0)                                # [P]

    # ----- in-flight fleet term (new) -------------------------------------
    f_alive = obs.f_alive
    if bool(f_alive.any()):
        f_owner = obs.f_owner.to(torch.long)
        f_enemy = f_alive & (f_owner >= 0) & (f_owner != pid)            # [F]
        if bool(f_enemy.any()):
            fx = obs.f_x.to(dtype)[f_enemy]                              # [E]
            fy = obs.f_y.to(dtype)[f_enemy]
            fs = obs.f_ships.to(dtype)[f_enemy].clamp(min=1e-6)          # [E]
            f_speed = fleet_speed(fs)                                    # [E]
            f_reach = (f_speed * H).clamp(min=1e-6)                      # [E]

            tx = obs.x.to(dtype).view(1, P)                              # [1, P]
            ty = obs.y.to(dtype).view(1, P)
            dxe = fx.view(-1, 1) - tx                                    # [E, P]
            dye = fy.view(-1, 1) - ty
            d_ft = torch.sqrt((dxe * dxe + dye * dye).clamp(min=0.0))    # [E, P]
            decay_f = (1.0 - d_ft / f_reach.view(-1, 1)).clamp(min=0.0)  # [E, P]
            tgt_alive = obs.alive.view(1, P)                             # [1, P]
            decay_f = torch.where(tgt_alive, decay_f, torch.zeros_like(decay_f))
            contrib_fleets = fs.view(-1, 1) * decay_f                    # [E, P]
            pressure = pressure + contrib_fleets.sum(dim=0)

    return pressure


def plan_lite_waves(
    *,
    movement: PlanetMovement,
    obs,
    obs_tensors: dict,
    cache,
    garrison_status,
    prod: Tensor,
    alive_by_step: Tensor,
    config: ProducerLiteConfig,
    player_count: int,
):
    """Multi-size, single-source attack planner + regroup.

    Builds multiple fleet-size candidates per ``(source, target)`` shortlist pair,
    scores them with the
    exact competitive flow diff, and greedily fires the best wave per target up to
    ``max_waves_per_turn``. Returns the combined ``LaunchEntries`` (attack waves ++
    regroup).
    """
    P = obs.P
    device = obs.device
    dtype = obs.ships.dtype
    pid = int(obs.player_id)

    H_axis = int(garrison_status.ships.shape[-1])
    H = max(H_axis - 1, 0)
    K_eta = max(1, min(int(config.horizon), H))
    W = max(1, int(config.max_waves_per_turn))

    source_mask = obs.owned & obs.alive & (obs.ships >= float(config.min_ships_to_launch))
    if not bool(source_mask.any()):
        return _empty_entries(device, dtype)

    S_cap = max(1, min(int(config.max_sources_per_lane), P))
    source_idx, source_exists = _candidate_indices(obs.ships, source_mask, S_cap)
    target_idx, target_exists = build_target_shortlist(
        obs, obs_tensors, garrison_status, cache,
        config=config, K_eta=K_eta, H=H, prod=prod, source_mask=source_mask,
    )
    if not bool(target_exists.any()):
        return _empty_entries(device, dtype)
    S = int(source_idx.shape[0])
    T = int(target_idx.shape[0])
    target_is_mine = obs.owned[target_idx.clamp(0, P - 1)]                       # [T]
    target_owner = obs.owner_abs[target_idx.clamp(0, P - 1)].to(torch.long)       # [T]
    target_prod = prod[target_idx.clamp(0, P - 1)].to(dtype)                      # [T]

    source_ships = obs.ships[source_idx.clamp(0, P - 1)].to(dtype)                # [S]
    H_eff = torch.full((), float(H), dtype=dtype, device=device)
    drain = safe_drain(
        garrison_status, source_idx=source_idx, source_ships=source_ships,
        H_eff=H_eff, player_id=pid,
    )                                                                            # [S]

    # Uniform reach cap = K_eta (= horizon).
    eta_cap = torch.full((T,), float(K_eta), dtype=dtype, device=device)          # [T]

    floor = capture_floor(
        garrison_status, target_idx=target_idx, k_max=K_eta,
        capture_overhead=1.0, player_id=pid,
    )                                                                            # [T, K]
    K = int(floor.shape[-1])

    # --- fleet-size ladder per (source, target) ---------------------------------
    R = 5
    drain_st = drain.view(S, 1).expand(S, T)                                      # [S, T]
    drain_str = drain_st.unsqueeze(-1).expand(S, T, R)                            # [S, T, R]

    # Strict-superset reachability precheck (always on): defers the body screen to
    # candidates that can physically reach the target in time.
    active = reachable_mask(
        movement, source_idx=source_idx, target_idx=target_idx,
        fleet_sizes=drain_st.unsqueeze(-1), eta_cap=eta_cap,
    ).squeeze(-1)                                                                # [S, T]
    aim = intercept_angle(
        movement,
        source_idx.unsqueeze(1),                                                 # [S, 1]
        target_idx.unsqueeze(0),                                                 # [1, T]
        drain_st,                                                                 # [S, T]
        active=active,
    )
    eta_base = aim["eta"]                                                       # [S, T]

    # Capture-floor gate at each fleet's arrival turn (defenders grow with k). The
    # single size must clear the defender it lands on (size >= floor_at_arr). Owned
    # targets have floor 1 (reinforcement), so any positive send clears.
    if K > 0:
        k_arr = (eta_base.clamp(min=1.0, max=float(K)).ceil().long() - 1).clamp(0, K - 1)  # [S,T]
        floor_seed_at_arr = floor.unsqueeze(0).expand(S, T, K).gather(-1, k_arr.unsqueeze(-1)).squeeze(-1)
    else:
        floor_seed_at_arr = torch.ones(S, T, dtype=dtype, device=device)
    raw_sizes = torch.stack(
        (
            floor_seed_at_arr + 1.0,
            floor_seed_at_arr + 3.0,
            0.35 * drain_st,
            0.65 * drain_st,
            drain_st,
        ),
        dim=-1,
    )                                                                            # [S, T, R]
    sizes = raw_sizes.floor().clamp(min=1.0)
    sizes = torch.minimum(sizes, drain_str.floor())                              # [S, T, R]
    size_seed_valid = torch.stack(
        (
            floor_seed_at_arr <= drain_st,
            floor_seed_at_arr <= drain_st,
            torch.ones_like(drain_st, dtype=torch.bool),
            torch.ones_like(drain_st, dtype=torch.bool),
            torch.ones_like(drain_st, dtype=torch.bool),
        ),
        dim=-1,
    )                                                                            # [S, T, R]
    active = active.unsqueeze(-1).expand(S, T, R)
    aim = intercept_angle(
        movement,
        source_idx.view(S, 1, 1).expand(S, T, R),
        target_idx.view(1, T, 1).expand(S, T, R),
        sizes,
        active=active,
    )
    angle = aim["angle"]                                                         # [S, T, R]
    eta = aim["eta"]
    viable = aim["viable"] & (eta <= eta_cap.view(1, T, 1))
    if K > 0:
        k_arr = (eta.clamp(min=1.0, max=float(K)).ceil().long() - 1).clamp(0, K - 1)  # [S,T,R]
        floor_at_arr = floor.unsqueeze(0).unsqueeze(2).expand(S, T, R, K).gather(-1, k_arr.unsqueeze(-1)).squeeze(-1)
    else:
        floor_at_arr = torch.ones(S, T, R, dtype=dtype, device=device)
    clears_floor = sizes >= floor_at_arr                                         # [S, T, R]

    src_neq_tgt = source_idx.view(S, 1) != target_idx.view(1, T)
    valid = (
        viable & clears_floor & size_seed_valid & (sizes >= 1.0) & src_neq_tgt.unsqueeze(-1)
        & source_exists.view(S, 1, 1) & target_exists.view(1, T, 1)
    )                                                                            # [S, T, R]

    # --- pack one candidate per (source, target, size); contributor axis L = 1 --
    L = 1
    C = S * T * R
    cand_src = source_idx.view(S, 1, 1).expand(S, T, R).reshape(C, L)
    cand_tgt_slot = target_idx.view(1, T, 1).expand(S, T, R).reshape(C)
    cand_tgt_short = torch.arange(T, device=device).view(1, T, 1).expand(S, T, R).reshape(C)
    cand_send = torch.where(valid, sizes, torch.zeros_like(sizes)).reshape(C, L)
    cand_angle = angle.reshape(C, L)
    cand_eta = torch.where(valid, eta, torch.ones_like(eta)).reshape(C, L)
    cand_active = valid.reshape(C, L)
    cand_valid = valid.reshape(C)
    cand_is_def = target_is_mine[cand_tgt_short]                                  # [C]
    cand_send_flat = cand_send.squeeze(-1)                                        # [C]
    cand_eta_flat = cand_eta.squeeze(-1)                                          # [C]
    cand_floor_at_arr = floor_at_arr.reshape(C)                                  # [C]
    cand_target_owner = target_owner[cand_tgt_short]                              # [C]
    cand_target_prod = target_prod[cand_tgt_short]                                # [C]

    launches = make_launch_set(
        source_slots=cand_src,
        target_slots=cand_tgt_slot.unsqueeze(-1).expand(C, L),
        ships=cand_send,
        eta=cand_eta,
        valid=cand_active & cand_valid.unsqueeze(-1),
        player_id=pid,
    )
    score = score_candidates(
        garrison_status, prod=prod, alive_by_step=alive_by_step,
        player_count=int(player_count), launches=launches, player_id=pid,
    )                                                                            # [C]

    neutral_owner_id = -1
    need = cand_floor_at_arr + float(config.min_capture_margin)
    overkill = (cand_send_flat - need).clamp(min=0.0)
    neutral_mask = cand_target_owner == int(neutral_owner_id)
    attack_mask = ~cand_is_def

    attack_cost = (
        float(config.ship_cost_penalty) * cand_send_flat
        + float(config.overkill_penalty) * overkill
        + neutral_mask.to(dtype) * float(config.neutral_overkill_penalty) * overkill
    )
    defense_cost = float(config.defense_ship_cost_penalty) * cand_send_flat
    score = score - torch.where(cand_is_def, defense_cost, attack_cost)

    turn = obs_tensors.get("step", torch.zeros((), dtype=dtype, device=device)).to(dtype).reshape(-1)[0]
    early_limit = max(1, int(config.neutral_early_turn_limit))
    early_factor = ((float(early_limit) - turn) / float(early_limit)).clamp(min=0.0, max=1.0)
    owner_count = max(1, int(player_count), pid + 1)
    total_ships = torch.zeros(owner_count, dtype=dtype, device=device)
    planet_owner = obs.owner_abs.to(torch.long)
    planet_owned = obs.alive & (planet_owner >= 0) & (planet_owner < owner_count)
    if bool(planet_owned.any()):
        total_ships.scatter_add_(0, planet_owner[planet_owned], obs.ships.to(dtype)[planet_owned])
    if hasattr(obs, "f_owner"):
        f_owner = obs.f_owner.to(torch.long)
        f_alive = obs.f_alive if hasattr(obs, "f_alive") else torch.ones_like(f_owner, dtype=torch.bool)
        f_owned = f_alive & (f_owner >= 0) & (f_owner < owner_count)
        if bool(f_owned.any()):
            total_ships.scatter_add_(0, f_owner[f_owned], obs.f_ships.to(dtype)[f_owned])
    my_total = total_ships[pid] if pid < owner_count else torch.zeros((), dtype=dtype, device=device)
    enemy_total = total_ships.clone()
    if pid < owner_count:
        enemy_total[pid] = 0.0
    enemy_max = enemy_total.max() if owner_count > 1 else torch.zeros((), dtype=dtype, device=device)
    early_factor = torch.where(my_total < 0.75 * enemy_max, torch.zeros_like(early_factor), early_factor)

    neutral_attack = neutral_mask & attack_mask
    neutral_bonus = (
        float(config.neutral_rush_bonus)
        + float(config.neutral_prod_bonus) * cand_target_prod
        - float(config.neutral_eta_penalty) * cand_eta_flat
    )
    score = score + neutral_attack.to(dtype) * early_factor * neutral_bonus

    cand_valid = cand_valid & (cand_is_def | (cand_send_flat >= need))
    threshold = torch.where(
        cand_is_def,
        torch.full_like(score, 0.2),
        torch.full_like(score, float(config.roi_threshold)),
    )
    score = score - threshold
    score = torch.where(cand_valid, score, torch.full_like(score, float("-inf")))

    safe_budget = torch.zeros_like(obs.ships.to(dtype))
    safe_budget[source_idx.clamp(0, P - 1)] = drain.floor()

    wave_entries, leftover = _greedy_select(
        P=P, W=W, device=device, dtype=dtype, score=score,
        cand_src=cand_src, cand_send=cand_send, cand_angle=cand_angle, cand_eta=cand_eta,
        cand_active=cand_active, cand_tgt_slot=cand_tgt_slot, cand_tgt_short=cand_tgt_short,
        cand_is_def=cand_is_def, source_budget=safe_budget,
        target_exists=target_exists, roi_threshold=0.0,
    )

    if not bool(config.enable_regroup):
        return wave_entries
    enemy_mass = cheap_enemy_pressure(obs, cache, horizon=float(K_eta), player_id=pid)  # [P]
    regroup_entries = _plan_regroup(
        movement=movement, obs=obs, obs_tensors=obs_tensors, garrison_status=garrison_status,
        leftover=leftover, original_ships=obs.ships.to(dtype), pressure=enemy_mass,
        config=config, H=H,
    )
    return concat_launch_entries([wave_entries, regroup_entries])


def run_turn(obs_tensors: dict, *, config: ProducerLiteConfig, player_count: int, memory) -> dict:
    """Full per-turn pipeline: build movement → plan multi-size waves + regroup → emit.

    ``memory`` must expose a mutable ``movement`` attribute (the rolling cache).
    """
    device = obs_tensors["planets"].device
    obs = parse_obs(obs_tensors)
    _debug_print_fleet_fields(obs, obs_tensors)
    P = obs.P
    if P == 0:
        return empty_action_row(device)

    movement = ensure_planet_movement(
        obs_tensors=obs_tensors,
        expected_cfg=_movement_config(config, player_count=int(player_count)),
        cached_movement=getattr(memory, "movement", None),
    )
    memory.movement = movement
    cache = build_distance_cache(movement, max_k=int(config.horizon))
    H = int(config.horizon)
    status = movement.garrison_status(max_horizon=H)
    alive_by_step = movement.alive_by_step[: H + 1]

    entries = plan_lite_waves(
        movement=movement, obs=obs, obs_tensors=obs_tensors, cache=cache,
        garrison_status=status, prod=movement.planet_prod,
        alive_by_step=alive_by_step, config=config, player_count=int(player_count),
    )
    entries = disambiguate_duplicate_launches(entries)
    launches = infer_planned_launches_from_entries(
        obs_tensors=obs_tensors, movement=movement, entries=entries, player_id=int(obs.player_id),
    )
    apply_private_planned_launches(
        movement=movement, launches=launches, owner_id=int(obs.player_id),
        obs_tensors=obs_tensors,
    )
    planet_ids = obs_tensors["planets"][..., 0].long()
    return entries_to_sparse_payload(entries, planet_ids=planet_ids)


# 4P FFA preset — only the knobs that differ from the 2P default. 
CONFIG_4P = dataclasses.replace(
    ProducerLiteConfig(),
    horizon=15,
    max_sources_per_lane=8,
    max_offensive_targets=14,
    max_defensive_targets=3,
    max_waves_per_turn=8,
    roi_threshold=1.8,
    max_regroup_time=6.0,
    max_regroup_targets_per_source=8,
)


def _config_for(player_count: int) -> ProducerLiteConfig:
    return CONFIG_4P if int(player_count) >= 4 else ProducerLiteConfig()


class ProducerLiteMemory:
    def __init__(self) -> None:
        self.movement = None
        self.cached_player_count: int | None = None
        self.last_sparse_action_row: dict | None = None

    def reset(self) -> None:
        self.movement = None
        self.cached_player_count = None
        self.last_sparse_action_row = None


class ProducerLiteRuntime:
    def __init__(self, memory: ProducerLiteMemory | None = None) -> None:
        self.memory = memory if memory is not None else ProducerLiteMemory()

    def reset(self) -> None:
        self.memory.reset()

    def tensor_action(self, obs_tensors: dict):
        mem = self.memory
        if bool((obs_tensors["step"] == 0).all()):
            mem.cached_player_count = None
        if mem.cached_player_count is None:
            mem.cached_player_count = largest_initial_player_count(obs_tensors)
        config = _config_for(mem.cached_player_count)
        row = run_turn(
            obs_tensors, config=config,
            player_count=int(mem.cached_player_count), memory=mem,
        )
        mem.last_sparse_action_row = row
        return row


_RUNTIME = ProducerLiteRuntime()


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def agent(obs):
    """Single-observation entry point for local play and Kaggle."""
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    player_id = int(player)
    obs_tensors = single_obs_to_tensor(obs, player_id=player_id)
    with torch.no_grad():
        sparse_row = _RUNTIME.tensor_action(obs_tensors)
    return sparse_action_row_to_moves(sparse_row, obs, player_id=player_id)



Writing main.py


In [ ]:
from pathlib import Path
import re

base = Path('/kaggle/input/datasets/slawekbiel/producer-orbit-wars-utils/orbit_lite')
print('[orbit_probe_source] begin orbit_lite source probe', flush=True)
print(f'[orbit_probe_source] base_exists={base.exists()} base={base}', flush=True)
patterns = re.compile(r'parse_obs|f_owner|f_ships|f_alive|f_.*target|target.*fleet|fleet.*target|fleets|fleet', re.I)
if base.exists():
    printed = 0
    stop = False
    for path in sorted(base.rglob('*.py')):
        if stop:
            break
        try:
            lines = path.read_text(errors='replace').splitlines()
        except Exception as exc:
            print(f'[orbit_probe_source] read_failed {path}: {type(exc).__name__}: {exc}', flush=True)
            continue
        hits = [i for i, line in enumerate(lines) if patterns.search(line)]
        if not hits:
            continue
        print(f'[orbit_probe_source] file={path.relative_to(base)} hits={len(hits)}', flush=True)
        windows = []
        for i in hits[:12]:
            windows.extend(range(max(0, i - 3), min(len(lines), i + 6)))
        for i in sorted(set(windows))[:180]:
            print(f'[orbit_probe_source] {path.relative_to(base)}:{i + 1}: {lines[i]}', flush=True)
            printed += 1
            if printed >= 700:
                print('[orbit_probe_source] output_cap_reached', flush=True)
                stop = True
                break
print('[orbit_probe_source] end orbit_lite source probe', flush=True)


In [2]:
!mkdir -p build
!mv main.py build/
!cp -r /kaggle/input/datasets/slawekbiel/producer-orbit-wars-utils/orbit_lite build/
!tar -czf submission.tar.gz -C build main.py orbit_lite
!rm -fR build